# Unlock Wallet Clustering

Explore clusters of wallets around high-inflow recipients derived from unlock CSV exports. Adjust the parameters below to control the inflow threshold, BFS depth, and optional CEX address monitoring.


In [86]:
from __future__ import annotations

import csv
import json
from collections import defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Set, Tuple

try:
    import pandas as pd  # type: ignore
except ImportError:
    pd = None

def resolve_data_dir() -> Path:
    search = [Path.cwd()] + list(Path.cwd().parents)
    for base in search:
        if (base / 'sui_insiders' / 'data').exists():
            return (base / 'sui_insiders' / 'data').resolve()
        if base.name == 'sui_insiders' and (base / 'data').exists():
            return (base / 'data').resolve()
        if (base / 'data').exists() and (base / 'scripts').exists():
            return (base / 'data').resolve()
    return (Path.cwd() / 'data').resolve()

DATA_DIR = resolve_data_dir()
DATA_DIR.mkdir(parents=True, exist_ok=True)

MIN_INFLOW_SUI = 1000.0
ROW_INFLOW_THRESHOLD = 1000.0  # per-transfer threshold for quick filtering
MAX_DEPTH = 2
CLUSTER_LIMIT = None  # set to an int to cap cluster size
UNDIRECTED_GRAPH = True
CEX_ADDRESS_FILE: Optional[Path] = None  # e.g., Path('cex_addresses.json')


In [87]:
@dataclass
class Transfer:
    event_label: str
    tx_digest: str
    timestamp_ms: int
    sender: str
    recipient: str
    amount_sui: float

def load_transfers(data_dir: Path) -> List[Transfer]:
    transfers: List[Transfer] = []
    for csv_path in sorted(data_dir.glob('*.csv')):
        with csv_path.open('r', encoding='utf-8') as fh:
            reader = csv.DictReader(fh)
            for row in reader:
                try:
                    amount_sui = float(row.get('amount_sui', 0))
                except (TypeError, ValueError):
                    amount_sui = 0.0
                transfers.append(
                    Transfer(
                        event_label=row.get('event_label') or '',
                        tx_digest=row.get('tx_digest') or '',
                        timestamp_ms=int(row.get('timestamp_ms') or 0),
                        sender=row.get('sender_address') or '',
                        recipient=row.get('recipient_address') or '',
                        amount_sui=amount_sui,
                    )
                )
    return transfers

def load_cex_addresses(path: Optional[Path]) -> Set[str]:
    if path is None:
        return set()
    data = json.loads(path.read_text(encoding='utf-8'))
    if isinstance(data, list):
        return {str(item) for item in data}
    if isinstance(data, dict):
        addresses: Set[str] = set()
        for value in data.values():
            if isinstance(value, list):
                addresses.update(str(item) for item in value)
        return addresses
    raise ValueError('Unsupported CEX address file format.')


In [88]:
TRANSFERS = load_transfers(DATA_DIR)
print(f'Loaded {len(TRANSFERS)} transfers from {DATA_DIR}')
if not TRANSFERS:
    raise RuntimeError('No transfer data available. Generate unlock CSVs first.')
CEX_ADDRESSES = load_cex_addresses(CEX_ADDRESS_FILE)
print(f'CEX address count: {len(CEX_ADDRESSES)}')


Loaded 20897 transfers from /Users/kevincai/Library/Mobile Documents/com~apple~CloudDocs/Career/Interviews/scimitar_capital/sui_insiders/data
CEX address count: 0


In [89]:
HIGH_INFLOW_OUTPUT = DATA_DIR / 'high_inflow_transfers.csv'

def filter_high_inflow_rows(threshold: float) -> tuple[list[dict[str, object]], list[str]]:
    rows: list[dict[str, object]] = []
    columns: list[str] | None = None
    for csv_path in sorted(DATA_DIR.glob('*.csv')):
        if csv_path.name == HIGH_INFLOW_OUTPUT.name:
            continue
        with csv_path.open('r', encoding='utf-8') as fh:
            reader = csv.DictReader(fh)
            if columns is None:
                columns = list(reader.fieldnames or [])
            for row in reader:
                try:
                    amount = float(row.get('amount_sui', 0))
                except (TypeError, ValueError):
                    amount = 0.0
                if amount >= threshold:
                    rows.append(row)
    return rows, (columns or ['event_label','tx_digest','checkpoint','timestamp_ms','sender_address','recipient_address','amount_mist','amount_sui'])

HIGH_INFLOW_ROWS, HIGH_INFLOW_COLUMNS = filter_high_inflow_rows(ROW_INFLOW_THRESHOLD)
with HIGH_INFLOW_OUTPUT.open('w', newline='', encoding='utf-8') as fh:
    writer = csv.DictWriter(fh, fieldnames=HIGH_INFLOW_COLUMNS)
    writer.writeheader()
    if HIGH_INFLOW_ROWS:
        writer.writerows(HIGH_INFLOW_ROWS)
print(f'Rows with amount_sui >= {ROW_INFLOW_THRESHOLD}: {len(HIGH_INFLOW_ROWS)}')
print(f'Saved filtered rows to {HIGH_INFLOW_OUTPUT}')
if HIGH_INFLOW_ROWS:
    if pd is not None:
        display(pd.DataFrame(HIGH_INFLOW_ROWS))
    else:
        for row in HIGH_INFLOW_ROWS:
            print(row)
else:
    print('No rows met the per-transfer threshold.')


ValueError: dict contains fields not in fieldnames: 'source_csv', 'seed_wallet', 'sale_type', 'timestamp', 'target_label'

In [90]:
def aggregate_inflows(transfers: Sequence[Transfer]) -> Dict[str, float]:
    totals: Dict[str, float] = defaultdict(float)
    for tx in transfers:
        if tx.recipient and tx.amount_sui > 0:
            totals[tx.recipient] += tx.amount_sui
    return totals

INFLOW_TOTALS = aggregate_inflows(TRANSFERS)
SEED_WALLETS = [wallet for wallet, total in INFLOW_TOTALS.items() if total >= MIN_INFLOW_SUI]
print(f'Seed wallets meeting inflow threshold ({MIN_INFLOW_SUI} SUI): {len(SEED_WALLETS)}')
if not SEED_WALLETS:
    print('No wallets met the threshold. Lower MIN_INFLOW_SUI or widen the unlock window.')
elif pd is not None:
    seed_df = pd.DataFrame([{'wallet': wallet, 'total_inflow_sui': INFLOW_TOTALS[wallet]} for wallet in SEED_WALLETS])
    seed_df = seed_df.sort_values('total_inflow_sui', ascending=False)
    display(seed_df)
else:
    for wallet in SEED_WALLETS:
        print(wallet, INFLOW_TOTALS[wallet])


Seed wallets meeting inflow threshold (1000.0 SUI): 165


,wallet,total_inflow_sui
85,0xecad8b6df613da99ebb12a2d42dffc61503628f0170d...,1.165841e+07
147,0x1c9deef7e10b88cca4211e5d688bb496d6843d916948...,3.962700e+06
28,0x344fb28d6b69274ec2ca62ccd40ef03eb01cec68495d...,3.718354e+06
88,0xcf30da961e1d310546f4df6dcbac0ace091852c77ed2...,3.637460e+06
29,0xdbc04ab99254d131f54d1e6d37397f0c4e3bb58a93eb...,3.601489e+06
...,...,...
16,0xf8b7b95d01ae79756fc3d1bc58675e59a17f07cdafe3...,1.304788e+03
76,0x77c389fea96d273b0b48d7bac4a7b51fcb60c5eb4530...,1.243091e+03
3,0x676087d52dfdcb981202d6f129f0259bdc5f3a1c90a4...,1.132332e+03
19,0x6cd00b0ccb13fe511776c8936af4ae74360de2f475d6...,1.024066e+03


In [91]:
def build_adjacency(transfers: Sequence[Transfer], undirected: bool) -> Dict[str, Set[str]]:
    graph: Dict[str, Set[str]] = defaultdict(set)
    for tx in transfers:
        if not tx.sender or not tx.recipient:
            continue
        graph[tx.sender].add(tx.recipient)
        if undirected:
            graph[tx.recipient].add(tx.sender)
    return graph

ADJACENCY = build_adjacency(TRANSFERS, undirected=UNDIRECTED_GRAPH)
len(ADJACENCY)


10400

In [92]:
def bfs_cluster(seed: str, adjacency: Dict[str, Set[str]], max_depth: int, limit: Optional[int] = None) -> Set[str]:
    visited: Set[str] = {seed}
    queue: deque[Tuple[str, int]] = deque([(seed, 0)])
    while queue:
        node, depth = queue.popleft()
        if depth >= max_depth:
            continue
        for neighbor in adjacency.get(node, set()):
            if neighbor in visited:
                continue
            visited.add(neighbor)
            queue.append((neighbor, depth + 1))
            if limit is not None and len(visited) >= limit:
                return visited
    return visited

def index_transfers_by_sender(transfers: Sequence[Transfer]) -> Dict[str, List[Transfer]]:
    by_sender: Dict[str, List[Transfer]] = defaultdict(list)
    for tx in transfers:
        if tx.sender:
            by_sender[tx.sender].append(tx)
    return by_sender

TRANSFERS_BY_SENDER = index_transfers_by_sender(TRANSFERS)


In [93]:
def summarize_cluster(cluster: Set[str]) -> Dict[str, object]:
    direct_hits: List[Dict[str, object]] = []
    total_direct = 0.0
    for wallet in cluster:
        for tx in TRANSFERS_BY_SENDER.get(wallet, []):
            if tx.recipient in CEX_ADDRESSES and tx.amount_sui > 0:
                direct_hits.append({
                    'sender': wallet,
                    'cex_recipient': tx.recipient,
                    'amount_sui': tx.amount_sui,
                    'tx_digest': tx.tx_digest,
                    'event_label': tx.event_label,
                })
                total_direct += tx.amount_sui
    return {
        'cluster_size': len(cluster),
        'wallets': sorted(cluster),
        'total_cluster_inflow_sui': sum(INFLOW_TOTALS.get(wallet, 0.0) for wallet in cluster),
        'direct_cex_transfers': direct_hits,
        'direct_cex_total_sui': total_direct,
    }


In [94]:
CLUSTER_RESULTS: Dict[str, Dict[str, object]] = {}
for wallet in SEED_WALLETS:
    cluster = bfs_cluster(wallet, ADJACENCY, max_depth=MAX_DEPTH, limit=CLUSTER_LIMIT)
    CLUSTER_RESULTS[wallet] = summarize_cluster(cluster)
len(CLUSTER_RESULTS)


165

In [95]:
def display_cluster_summary(results: Dict[str, Dict[str, object]]) -> None:
    rows = []
    for seed, summary in results.items():
        rows.append({
            'seed_wallet': seed,
            'cluster_size': summary['cluster_size'],
            'total_cluster_inflow_sui': summary['total_cluster_inflow_sui'],
            'direct_cex_total_sui': summary['direct_cex_total_sui'],
            'direct_cex_tx_count': len(summary['direct_cex_transfers']),
        })
    if pd is not None and rows:
        df = pd.DataFrame(rows).sort_values('total_cluster_inflow_sui', ascending=False)
        display(df)
    else:
        for row in rows:
            print(row)

display_cluster_summary(CLUSTER_RESULTS)


,seed_wallet,cluster_size,total_cluster_inflow_sui,direct_cex_total_sui,direct_cex_tx_count
88,0xcf30da961e1d310546f4df6dcbac0ace091852c77ed2...,3,1.615770e+07,0.0,0
84,0x63fbe037f5cca825dbb66db0464b34ece8ad64413cc0...,3,1.615770e+07,0.0,0
85,0xecad8b6df613da99ebb12a2d42dffc61503628f0170d...,3,1.615770e+07,0.0,0
8,0xab73ad38c63f83eda02182422b545395be1d3caeb54b...,81,4.431998e+06,0.0,0
38,0x8b302e06dd0bc3e09f1e009e1dc32a04f6b4b75ab550...,31,4.360950e+06,0.0,0
...,...,...,...,...,...
74,0xb53a1af6c90f0f023bf93308ea87a934267933ff9cf9...,1,2.213814e+03,0.0,0
13,0x570f39c18f08d5fad6de7dff90a9477c5046d4d0b671...,1,2.063879e+03,0.0,0
106,0x41c4ef00a0b21a25f4fe746aceb2a458109fb4822083...,1,2.033765e+03,0.0,0
19,0x6cd00b0ccb13fe511776c8936af4ae74360de2f475d6...,3,1.576047e+03,0.0,0


In [96]:
def inspect_cluster(seed_wallet: str) -> None:
    summary = CLUSTER_RESULTS.get(seed_wallet)
    if summary is None:
        print(f'No cluster data found for {seed_wallet}')
        return
    print(f'=== {seed_wallet} ===')
    print(f"Cluster size: {summary['cluster_size']}, total inflow: {summary['total_cluster_inflow_sui']:.2f} SUI")
    if summary['direct_cex_transfers']:
        print(f"Direct CEX transfers ({summary['direct_cex_total_sui']:.2f} SUI):")
        for hit in summary['direct_cex_transfers']:
            print('  ->', hit)
    else:
        print('No direct CEX transfers detected.')
    print('Wallets in cluster:')
    for wallet in summary['wallets']:
        print(' ', wallet)

# Example usage:
# inspect_cluster(SEED_WALLETS[0])


In [97]:
# Export cluster summaries to CSV and JSON for external analysis.
CLUSTER_OUTPUT_JSON = DATA_DIR / 'cluster_results.json'
CLUSTER_OUTPUT_CSV = DATA_DIR / 'cluster_results.csv'
CLUSTER_TX_CSV = DATA_DIR / 'cluster_cex_transfers.csv'

CLUSTER_OUTPUT_JSON.write_text(json.dumps(CLUSTER_RESULTS, indent=2), encoding='utf-8')
print(f'Saved cluster JSON to {CLUSTER_OUTPUT_JSON}')

summary_fields = ['seed_wallet', 'cluster_size', 'total_cluster_inflow_sui', 'direct_cex_total_sui', 'direct_cex_tx_count']
with CLUSTER_OUTPUT_CSV.open('w', newline='', encoding='utf-8') as fh:
    writer = csv.DictWriter(fh, fieldnames=summary_fields)
    writer.writeheader()
    for seed, summary in CLUSTER_RESULTS.items():
        row = {
            'seed_wallet': seed,
            'cluster_size': summary['cluster_size'],
            'total_cluster_inflow_sui': summary['total_cluster_inflow_sui'],
            'direct_cex_total_sui': summary['direct_cex_total_sui'],
            'direct_cex_tx_count': len(summary['direct_cex_transfers']),
        }
        writer.writerow(row)
print(f'Saved cluster summary CSV to {CLUSTER_OUTPUT_CSV}')

cex_fields = ['seed_wallet', 'sender', 'cex_recipient', 'amount_sui', 'tx_digest', 'event_label']
with CLUSTER_TX_CSV.open('w', newline='', encoding='utf-8') as fh:
    writer = csv.DictWriter(fh, fieldnames=cex_fields)
    writer.writeheader()
    for seed, summary in CLUSTER_RESULTS.items():
        for hit in summary['direct_cex_transfers']:
            writer.writerow({
                'seed_wallet': seed,
                'sender': hit['sender'],
                'cex_recipient': hit['cex_recipient'],
                'amount_sui': hit['amount_sui'],
                'tx_digest': hit['tx_digest'],
                'event_label': hit['event_label'],
            })
print(f'Saved CEX transfer CSV to {CLUSTER_TX_CSV}')


Saved cluster JSON to /Users/kevincai/Library/Mobile Documents/com~apple~CloudDocs/Career/Interviews/scimitar_capital/sui_insiders/data/cluster_results.json
Saved cluster summary CSV to /Users/kevincai/Library/Mobile Documents/com~apple~CloudDocs/Career/Interviews/scimitar_capital/sui_insiders/data/cluster_results.csv
Saved CEX transfer CSV to /Users/kevincai/Library/Mobile Documents/com~apple~CloudDocs/Career/Interviews/scimitar_capital/sui_insiders/data/cluster_cex_transfers.csv
